# Data Analyse

Shows a finished token-analysis run: country and global token statistics, the summary report and its charts.

This notebook builds nothing. A run is made by `scripts/analyze_tokens.py`, which writes it to `artifacts/analysis/token_performance/runs/<run_date>/`; this notebook finds the runs on disk, picks one, and reads it back. `analysis.token_metrics` names every file, so nothing here spells a path.

If there is no run yet, make one with:

```bash
uv run --no-sync python scripts/analyze_tokens.py --systems ie
```


In [ ]:
import sys
from pathlib import Path

import polars as pl
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import _bootstrap  # noqa: F401

from analysis.token_metrics import resolve_phase1_paths
from workspace.artifact_layout import analysis_report_runs_root
from workspace.roots import default_workspace_roots

ROOTS = default_workspace_roots(ROOT)

# Local notebook configuration (edit directly in this cell).
RUN_DATE = None  # YYYY-MM-DD, or None for the latest run on disk

print(f"ROOT={ROOT}")

## 1) Find the run

Lists the dates a finished run exists for, and resolves the chosen run's files.


In [ ]:
# The report's runs sit one directory per run date; the name is token_metrics' own.
runs_root = analysis_report_runs_root(ROOTS, "token_performance")
run_dates = (
    sorted(path.name for path in runs_root.iterdir() if path.is_dir())
    if runs_root.is_dir()
    else []
)
if not run_dates:
    raise FileNotFoundError(
        f"No token-analysis run on disk under {runs_root}. Make one with:\n"
        "  uv run --no-sync python scripts/analyze_tokens.py --systems <code>"
    )
print("Run dates on disk:", ", ".join(run_dates))

run_date = RUN_DATE or run_dates[-1]
if run_date not in run_dates:
    raise ValueError(f"RUN_DATE={run_date!r} names none of the run dates listed above")

# The run's directories exist, so resolving its paths creates nothing.
paths = resolve_phase1_paths(ROOTS, run_date)
print(f"Run root: {paths.run_root}")
if paths.report_path.exists():
    display(Markdown(paths.report_path.read_text(encoding="utf-8")))

In [ ]:
country_df = pl.read_parquet(paths.country_stats_path)
global_df = pl.read_parquet(paths.global_stats_path)

print("Country rows:", country_df.height)
print("Global rows:", global_df.height)
display(country_df.head(20))
display(global_df.head(20))

for chart_path in (
    paths.overview_plot_path,
    paths.top_tokens_plot_path,
    paths.top_tokens_ex_legal_form_plot_path,
    paths.top_tfidf_plot_path,
    paths.top_tfidf_ex_legal_form_plot_path,
):
    if chart_path.exists():
        print(chart_path.name)
        display(Image(filename=str(chart_path)))